# COMBINACIÓN ARCHIVOS PROCESADOS DEFUNCIONES

## 0. LIMPIAR MEMORIA

In [1]:
# 0. LIMPIAR MEMORIA ANTES DE INICIAR
import os
import psutil
import gc
def limpiar_memoria():
    """Libera memoria RAM y limpia objetos no usados."""
    gc.collect()
    process = psutil.Process(os.getpid())
    memoria_libre = process.memory_info().rss / (1024 ** 3)
    print(f"🔄 Memoria usada antes de limpieza: {memoria_libre:.2f} GB")

    for var in list(globals().keys()):
        if not var.startswith("_") and var not in ["os", "gc", "psutil", "limpiar_memoria"]:
            del globals()[var]

    gc.collect()
    memoria_final = process.memory_info().rss / (1024 ** 3)
    print(f"✅ Memoria usada después de limpieza: {memoria_final:.2f} GB")
limpiar_memoria()

🔄 Memoria usada antes de limpieza: 0.07 GB
✅ Memoria usada después de limpieza: 0.07 GB


## 1. IMPORTAR LIBRERIAS

In [2]:
import pandas as pd
import os
import gc

## 2. DEFINIR FUNCIONES

### 2.1. Función para lectura de archivos

In [3]:
def cargar_archivos_seleccionados_con_id(ruta_carpeta, lista_archivos):
    """
    Lee los archivos .parquet especificados en 'lista_archivos' desde 'ruta_carpeta'.
    Asigna un ID incremental (1, 2, 3, ...) y crea variables globales
    con el mismo nombre del archivo (sin extensión).

    Si un archivo no se encuentra, muestra un mensaje de advertencia y continúa.
    """
    total_cargados = 0

    for idx, archivo in enumerate(lista_archivos, start=1):
        ruta = os.path.join(ruta_carpeta, archivo)
        
        # Validar existencia del archivo
        if not os.path.exists(ruta):
            print(f"⚠️ Archivo no encontrado: {archivo}")
            continue  # saltar al siguiente
        
        try:
            df = pd.read_parquet(ruta)
            df["ID"] = idx
            
            nombre_var = os.path.splitext(archivo)[0]
            globals()[nombre_var] = df  # crea variable global
            
            print(f"✅ {nombre_var} cargado con ID = {idx} ({len(df)} filas)")
            total_cargados += 1

        except Exception as e:
            print(f"❌ Error al leer {archivo}: {e}")
    
    print(f"\n📊 Archivos procesados exitosamente: {total_cargados} de {len(lista_archivos)}")


### 2.2. Función para mostrar primeros 5 registros de cada archivo cargado

In [4]:
def mostrar_head_archivos(lista_archivos):
    """
    Muestra las primeras 5 filas de cada DataFrame en memoria,
    basándose en los nombres de los archivos indicados en 'lista_archivos'.
    Si una variable no existe, muestra una advertencia y continúa.
    """
    # Mostrar todas las columnas en el output
    pd.set_option('display.max_columns', None)
    
    total_mostrados = 0

    for archivo in lista_archivos:
        nombre_var = os.path.splitext(archivo)[0]
        
        if nombre_var in globals():
            df = globals()[nombre_var]
            print(f"\n🧾 DataFrame: {nombre_var}")
            display(df.head(5))
            print("-" * 100)
            total_mostrados += 1
        else:
            print(f"⚠️ El DataFrame '{nombre_var}' no se encuentra cargado en memoria.")
    
    print(f"\n📊 Total de DataFrames mostrados: {total_mostrados} de {len(lista_archivos)}")

### 2.3. Función para cargar diccionario de homologacion

In [5]:
def cargar_diccionario_homologacion(ruta_excel, nombre_hoja="Campos Defunciones"):
    """
    Lee un archivo Excel que contiene la homologación de campos.
    Para cada campo (columna), detecta el ÚLTIMO nombre disponible (no nulo)
    y lo usa como nombre estándar.
    
    VALIDACIÓN: Solo procesa filas con ID válido Y al menos un campo con nombre.
    
    Parámetros:
    -----------
    ruta_excel : str
        Ruta completa al archivo Excel de homologación
    nombre_hoja : str
        Nombre de la hoja a leer (por defecto "Campos Defunciones")
    
    Retorna:
    --------
    dict : Diccionario donde la clave es el ID y el valor es otro diccionario
           con los mapeos de columnas antiguas -> columnas nuevas
    dict : Diccionario con los nombres estándar por columna
    """
    try:
        # Leer Excel
        df_homolog = pd.read_excel(ruta_excel, sheet_name=nombre_hoja)
        
        print(f"\n🔍 DEBUG - Información del Excel:")
        print(f"   Dimensiones originales: {df_homolog.shape}")
        print(f"   Columnas totales: {len(df_homolog.columns)}")
        
        # Verificar que existe columna ID
        if 'ID' not in df_homolog.columns:
            print("❌ ERROR: No se encontró la columna 'ID' en el Excel")
            return None, None
        
        # PASO 1: FILTRAR Y VALIDAR FILAS
        print(f"\n🔍 Validando filas del Excel...")
        
        # Filtrar filas que tengan ID válido (numérico y no nulo)
        df_homolog = df_homolog[df_homolog['ID'].notna()].copy()
        
        # Convertir ID a entero (esto también filtra IDs no numéricos)
        try:
            df_homolog['ID'] = pd.to_numeric(df_homolog['ID'], errors='coerce')
            df_homolog = df_homolog[df_homolog['ID'].notna()].copy()
            df_homolog['ID'] = df_homolog['ID'].astype(int)
        except Exception as e:
            print(f"❌ Error al procesar columna ID: {e}")
            return None, None
        
        # Obtener las columnas de campos (todas excepto 'ID')
        columnas_campos = [col for col in df_homolog.columns if col != 'ID']
        
        # VALIDACIÓN CRÍTICA: Solo mantener filas que tengan al menos UN campo con valor
        print(f"   Filas con ID válido: {len(df_homolog)}")
        
        # Contar valores no nulos en las columnas de campos
        df_homolog['_campos_validos'] = df_homolog[columnas_campos].notna().sum(axis=1)
        
        # Filtrar: solo filas con al menos 1 campo válido
        filas_con_datos = df_homolog[df_homolog['_campos_validos'] > 0].copy()
        filas_sin_datos = df_homolog[df_homolog['_campos_validos'] == 0]
        
        print(f"   ├─ Con datos en campos: {len(filas_con_datos)}")
        print(f"   └─ Sin datos (descartadas): {len(filas_sin_datos)}")
        
        if len(filas_sin_datos) > 0:
            ids_descartados = sorted(filas_sin_datos['ID'].tolist())
            if len(ids_descartados) <= 10:
                print(f"      IDs descartados: {ids_descartados}")
            else:
                print(f"      IDs descartados (primeros 10): {ids_descartados[:10]}")
                print(f"      ... y {len(ids_descartados) - 10} más")
        
        # Usar solo filas válidas
        df_homolog = filas_con_datos.drop('_campos_validos', axis=1)
        
        if len(df_homolog) == 0:
            print("❌ ERROR: No hay filas válidas para procesar")
            return None, None
        
        # Ordenar por ID
        df_homolog = df_homolog.sort_values('ID')
        
        print(f"\n   ✅ Filas válidas a procesar: {len(df_homolog)}")
        print(f"   📋 Rango de IDs: {int(df_homolog['ID'].min())} - {int(df_homolog['ID'].max())}")
        print(f"   📋 Campos a procesar: {len(columnas_campos)}")
        
        # PASO 2: DETECTAR NOMBRES ESTÁNDAR (último valor disponible por columna)
        print(f"\n🔍 Detectando nombres estándar (último valor disponible por columna)...")
        nombres_estandar = {}
        
        for campo in columnas_campos:
            # Obtener todos los valores no nulos de esta columna, en orden de ID
            valores_no_nulos = df_homolog[df_homolog[campo].notna()][campo].tolist()
            
            if valores_no_nulos:
                # El último valor no nulo es el nombre estándar
                ultimo_valor = str(valores_no_nulos[-1]).strip()
                if ultimo_valor:
                    nombres_estandar[campo] = ultimo_valor
                    # Mostrar solo algunos ejemplos
                    if len(nombres_estandar) <= 5:
                        print(f"   ✓ '{campo}' → '{ultimo_valor}'")
        
        if len(nombres_estandar) > 5:
            print(f"   ... y {len(nombres_estandar) - 5} campos más")
        
        print(f"\n   📊 Total nombres estándar detectados: {len(nombres_estandar)}")
        
        # PASO 3: CREAR DICCIONARIO DE HOMOLOGACIÓN POR ID
        print(f"\n🔄 Creando mapeos de homologación por ID...")
        diccionario = {}
        
        ids_procesados = []
        ids_sin_mapeos = []
        
        for idx, row in df_homolog.iterrows():
            id_actual = int(row['ID'])
            mapeo = {}
            
            # Para cada campo, mapear: nombre_actual → nombre_estandar
            for campo in columnas_campos:
                nombre_actual = row[campo]
                
                # Solo procesar si hay un nombre estándar definido para este campo
                if campo in nombres_estandar:
                    nombre_estandar = nombres_estandar[campo]
                    
                    # Si hay un valor actual (no nulo) y es diferente del estándar
                    if pd.notna(nombre_actual):
                        nombre_actual_str = str(nombre_actual).strip()
                        
                        if nombre_actual_str and nombre_actual_str != nombre_estandar:
                            mapeo[nombre_actual_str] = nombre_estandar
            
            diccionario[id_actual] = mapeo
            ids_procesados.append(id_actual)
            
            if len(mapeo) == 0:
                ids_sin_mapeos.append(id_actual)
        
        # REPORTE DE MAPEOS
        print(f"\n   ✅ IDs procesados: {len(ids_procesados)}")
        
        ids_con_mapeos = [id_val for id_val in ids_procesados if id_val not in ids_sin_mapeos]
        print(f"   ├─ Con mapeos (campos a renombrar): {len(ids_con_mapeos)}")
        print(f"   └─ Sin mapeos (nombres ya estándar): {len(ids_sin_mapeos)}")
        
        if ids_sin_mapeos:
            if len(ids_sin_mapeos) <= 10:
                print(f"      IDs sin mapeos: {ids_sin_mapeos}")
            else:
                print(f"      IDs sin mapeos (primeros 10): {ids_sin_mapeos[:10]}")
        
        # Mostrar ejemplos de mapeos
        print(f"\n   📋 Ejemplos de mapeos creados:")
        ejemplos_mostrados = 0
        for id_val in sorted(ids_con_mapeos)[:3]:
            if id_val in diccionario and diccionario[id_val]:
                print(f"      ID {id_val}: {len(diccionario[id_val])} campos a renombrar")
                # Mostrar 2 ejemplos de renombramientos
                ejemplos = list(diccionario[id_val].items())[:2]
                for orig, nuevo in ejemplos:
                    print(f"         '{orig}' → '{nuevo}'")
                ejemplos_mostrados += 1
                if ejemplos_mostrados >= 3:
                    break
        
        print(f"\n{'='*80}")
        print(f"✅ DICCIONARIO DE HOMOLOGACIÓN CARGADO EXITOSAMENTE")
        print(f"{'='*80}")
        print(f"📋 IDs válidos procesados: {len(diccionario)}")
        print(f"📋 Campos estándar definidos: {len(nombres_estandar)}")
        print(f"✅ Listo para homologación")
        print(f"{'='*80}\n")
        
        return diccionario, nombres_estandar
    
    except Exception as e:
        print(f"\n❌ Error al cargar el diccionario de homologación: {e}")
        import traceback
        print(f"📍 Detalles del error:")
        traceback.print_exc()
        return None, None

### 2.4. Función para homologar y combinar dataframes

In [6]:
def homologar_y_combinar_defunciones_optimizado(lista_dataframes, diccionario_homolog, nombres_estandar, 
                                                 ruta_salida="data/processed/defunciones_completo.parquet",
                                                 batch_size=3):
    """
    Versión optimizada para memoria: procesa y guarda por lotes.
    
    ESTRATEGIA:
    1. Procesa DataFrames en lotes pequeños
    2. Guarda resultados parciales
    3. Combina archivos parciales al final
    4. Reduce uso de RAM significativamente
    
    Parámetros:
    -----------
    batch_size : int
        Número de DataFrames a procesar simultáneamente (default: 3)
        Ajusta según tu RAM disponible
    """
    
    if diccionario_homolog is None:
        print("❌ No hay diccionario de homologación disponible")
        return None
    
    import gc
    
    # ========================================================================
    # FASE 0: ANÁLISIS PREVIO (sin cargar datos completos)
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🚀 PROCESAMIENTO OPTIMIZADO PARA MEMORIA")
    print(f"{'='*80}")
    print(f"📊 Configuración:")
    print(f"   • Total de DataFrames: {len(lista_dataframes)}")
    print(f"   • Tamaño de lote: {batch_size}")
    print(f"   • Lotes necesarios: {(len(lista_dataframes) + batch_size - 1) // batch_size}")
    
    # ========================================================================
    # PASO PREVIO: ANÁLISIS DE COLUMNAS Y MAPEO COMPLETO
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"📊 FASE 1: ANÁLISIS DE COLUMNAS Y CREACIÓN DE MAPEO UNIVERSAL")
    print(f"{'='*80}")
    
    # 1. Crear mapeo universal: nombre_original -> nombre_estándar
    mapeo_universal = {}  # {nombre_original: nombre_final}
    
    # Primero, agregar todos los mapeos del Excel
    for id_archivo, mapeos in diccionario_homolog.items():
        for nombre_original, nombre_estandar in mapeos.items():
            if nombre_original not in mapeo_universal:
                mapeo_universal[nombre_original] = nombre_estandar
            elif mapeo_universal[nombre_original] != nombre_estandar:
                # Conflicto: mismo nombre original mapea a diferentes estándares
                print(f"   ⚠️  CONFLICTO: '{nombre_original}' mapea a:")
                print(f"      • '{mapeo_universal[nombre_original]}' (anterior)")
                print(f"      • '{nombre_estandar}' (ID {id_archivo})")
                print(f"      → Manteniendo: '{mapeo_universal[nombre_original]}'")
    
    # 2. Agregar nombres del Excel que ya son estándar (identidad: nombre -> nombre)
    for nombre_estandar in nombres_estandar:
        if nombre_estandar not in mapeo_universal:
            mapeo_universal[nombre_estandar] = nombre_estandar
    
    # 3. Analizar todas las columnas en todos los DataFrames
    todas_columnas_originales = {}  # {nombre_columna: [lista de IDs donde aparece]}
    
    for nombre_var, id_archivo in lista_dataframes:
        if nombre_var not in globals():
            continue
        
        df_temp = globals()[nombre_var]
        for col in df_temp.columns:
            if col != 'ID':
                if col not in todas_columnas_originales:
                    todas_columnas_originales[col] = []
                todas_columnas_originales[col].append(id_archivo)
    
    # 4. Para columnas NO en el Excel, determinar su nombre final
    # Si una columna aparece en múltiples DataFrames, ese ES su nombre final
    for nombre_col, ids in todas_columnas_originales.items():
        if nombre_col not in mapeo_universal:
            # Esta columna no está en el Excel, usar su nombre tal cual
            mapeo_universal[nombre_col] = nombre_col
    
    # 5. Crear mapeo inverso para detectar duplicados
    # nombre_final -> [lista de nombres_originales que mapean a él]
    mapeo_inverso = {}
    for nombre_orig, nombre_final in mapeo_universal.items():
        if nombre_final not in mapeo_inverso:
            mapeo_inverso[nombre_final] = []
        if nombre_orig not in mapeo_inverso[nombre_final]:
            mapeo_inverso[nombre_final].append(nombre_orig)
    
    # 6. Detectar y reportar columnas con múltiples orígenes
    print(f"\n📋 MAPEO UNIVERSAL CREADO:")
    print(f"   • Nombres originales únicos: {len(mapeo_universal)}")
    print(f"   • Nombres finales únicos: {len(mapeo_inverso)}")
    
    columnas_multiples_origenes = {final: origenes for final, origenes in mapeo_inverso.items() if len(origenes) > 1}
    if columnas_multiples_origenes:
        print(f"\n   ✅ Columnas con múltiples nombres originales (se consolidarán):")
        for i, (nombre_final, nombres_orig) in enumerate(list(columnas_multiples_origenes.items())[:10], 1):
            print(f"      {i}. '{nombre_final}' ← {nombres_orig}")
        if len(columnas_multiples_origenes) > 10:
            print(f"      ... y {len(columnas_multiples_origenes) - 10} más")
    
    # 7. Determinar todas las columnas finales únicas
    todas_columnas_finales = sorted(list(set(mapeo_universal.values())))
    
    print(f"\n   📊 COLUMNAS FINALES: {len(todas_columnas_finales)}")
    
    # Clasificar columnas finales
    columnas_del_excel_final = [col for col in todas_columnas_finales if col in nombres_estandar]
    columnas_adicionales_final = [col for col in todas_columnas_finales if col not in nombres_estandar]
    
    print(f"      ✅ Del Excel: {len(columnas_del_excel_final)}")
    print(f"      ➕ Adicionales: {len(columnas_adicionales_final)}")
    
    # ========================================================================
    # FASE 2: PROCESAMIENTO POR LOTES
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🔄 FASE 2: PROCESAMIENTO POR LOTES")
    print(f"{'='*80}")
    
    archivos_parciales = []
    lote_actual = 0
    
    for i in range(0, len(lista_dataframes), batch_size):
        lote_actual += 1
        batch = lista_dataframes[i:i + batch_size]
        
        print(f"\n{'─'*80}")
        print(f"📦 LOTE {lote_actual}/{(len(lista_dataframes) + batch_size - 1) // batch_size}")
        print(f"   DataFrames: {i+1} a {min(i+batch_size, len(lista_dataframes))}")
        print(f"{'─'*80}")
        
        dfs_procesados = []
        
        for idx_global, (nombre_var, id_archivo) in enumerate(batch, start=i+1):
            if nombre_var not in globals():
                print(f"⚠️  [{idx_global}] '{nombre_var}' no encontrado")
                continue
            
            print(f"\n🔄 [{idx_global}/{len(lista_dataframes)}] {nombre_var} (ID: {id_archivo})")
            
            # Cargar DataFrame
            df = globals()[nombre_var].copy()
            print(f"   📏 Dimensiones originales: {df.shape}")
            print(f"   📋 Columnas originales: {len(df.columns)}")
            
            # Eliminar columna ID interna si existe
            if 'ID' in df.columns:
                df.drop('ID', axis=1, inplace=True)
            
            # ════════════════════════════════════════════════════════════════
            # PASO CLAVE: APLICAR MAPEO UNIVERSAL (no solo del ID actual)
            # ════════════════════════════════════════════════════════════════
            print(f"   🔄 Aplicando mapeo universal de nombres...")
            
            columnas_originales = list(df.columns)
            mapeo_aplicado = {}
            
            for col_original in columnas_originales:
                if col_original in mapeo_universal:
                    nombre_final = mapeo_universal[col_original]
                    if col_original != nombre_final:
                        mapeo_aplicado[col_original] = nombre_final
            
            if mapeo_aplicado:
                print(f"   ✅ Renombrando {len(mapeo_aplicado)} columnas:")
                # Mostrar algunos ejemplos
                ejemplos = list(mapeo_aplicado.items())[:5]
                for orig, final in ejemplos:
                    print(f"      • '{orig}' → '{final}'")
                if len(mapeo_aplicado) > 5:
                    print(f"      ... y {len(mapeo_aplicado) - 5} más")
                
                df.rename(columns=mapeo_aplicado, inplace=True)
            else:
                print(f"   ℹ️  No se requieren renombramientos")
            
            # ════════════════════════════════════════════════════════════════
            # VERIFICAR Y CONSOLIDAR COLUMNAS DUPLICADAS
            # ════════════════════════════════════════════════════════════════
            columnas_duplicadas = df.columns[df.columns.duplicated()].unique()
            if len(columnas_duplicadas) > 0:
                print(f"   ⚠️  Detectadas {len(columnas_duplicadas)} columnas duplicadas:")
                for col_dup in columnas_duplicadas:
                    print(f"      • '{col_dup}'")
                
                print(f"   🔧 Consolidando columnas duplicadas...")
                
                # Para cada columna duplicada, consolidar los valores
                for col_dup in columnas_duplicadas:
                    # Obtener todas las columnas con ese nombre
                    cols_indices = [i for i, col in enumerate(df.columns) if col == col_dup]
                    
                    if len(cols_indices) > 1:
                        # Crear nueva columna consolidada usando coalesce (primer valor no nulo)
                        columnas_a_combinar = [df.iloc[:, idx] for idx in cols_indices]
                        
                        # Combinar usando el primer valor no nulo de cada fila
                        import numpy as np
                        df_consolidado = pd.DataFrame(columnas_a_combinar).T
                        columna_consolidada = df_consolidado.apply(
                            lambda row: row.dropna().iloc[0] if row.notna().any() else None, 
                            axis=1
                        )
                        
                        # Eliminar las columnas duplicadas
                        df = df.drop(df.columns[cols_indices], axis=1)
                        
                        # Agregar la columna consolidada
                        df[col_dup] = columna_consolidada
                
                print(f"   ✅ Columnas consolidadas")
            
            print(f"   📋 Columnas después de renombrar: {len(df.columns)}")
            
            # AGREGAR columnas faltantes (todas las que existen en el universo)
            columnas_actuales = set(df.columns)
            columnas_faltantes = set(todas_columnas_finales) - columnas_actuales
            
            if columnas_faltantes:
                print(f"   ➕ Agregando {len(columnas_faltantes)} columnas faltantes")
                for col in columnas_faltantes:
                    if col not in df.columns:  # Verificación adicional
                        df[col] = None
            
            # VERIFICACIÓN FINAL: Asegurar que no hay duplicados antes de reordenar
            if df.columns.duplicated().any():
                print(f"   ⚠️  Aún hay duplicados, eliminando...")
                df = df.loc[:, ~df.columns.duplicated(keep='first')]
            
            # REORDENAR columnas alfabéticamente para consistencia
            # Usar solo columnas que existen en el DataFrame
            columnas_disponibles = [col for col in todas_columnas_finales if col in df.columns]
            df = df[columnas_disponibles]
            
            # OPTIMIZAR TIPOS DE DATOS (crucial para memoria)
            print(f"   🔧 Optimizando tipos de datos...")
            df = optimizar_tipos_dataframe(df)
            
            print(f"   ✅ Procesado completo: {df.shape}")
            
            dfs_procesados.append(df)
        
        # COMBINAR lote actual
        if dfs_procesados:
            print(f"\n🔗 Combinando {len(dfs_procesados)} DataFrames del lote...")
            
            # Verificar que todos los DataFrames tienen las mismas columnas
            print(f"   🔍 Verificando consistencia de columnas...")
            columnas_por_df = [set(df.columns) for df in dfs_procesados]
            
            # Encontrar columnas comunes
            columnas_comunes = set.intersection(*columnas_por_df) if columnas_por_df else set()
            print(f"   ✅ Columnas comunes: {len(columnas_comunes)}")
            
            # Verificar duplicados en cada DataFrame antes de concatenar
            for i, df in enumerate(dfs_procesados):
                if df.columns.duplicated().any():
                    print(f"   ⚠️  DataFrame {i+1} tiene columnas duplicadas")
                    dfs_procesados[i] = df.loc[:, ~df.columns.duplicated(keep='first')]
            
            df_lote = pd.concat(dfs_procesados, ignore_index=True, sort=False)
            
            # Eliminar duplicados de columnas en el resultado
            if df_lote.columns.duplicated().any():
                print(f"   ⚠️  Lote combinado tiene columnas duplicadas, eliminando...")
                df_lote = df_lote.loc[:, ~df_lote.columns.duplicated(keep='first')]
            
            print(f"   ✅ Lote combinado: {df_lote.shape}")
            
            # GUARDAR lote parcial
            archivo_parcial = f"data/processed/temp_lote_{lote_actual:03d}.parquet"
            os.makedirs("data/processed", exist_ok=True)
            
            print(f"   💾 Guardando lote parcial...")
            # Limpiar DataFrame antes de guardar
            df_lote_limpio = limpiar_dataframe_para_parquet(df_lote)
            df_lote_limpio.to_parquet(archivo_parcial, index=False, engine='pyarrow', compression='snappy')
            
            tamaño_mb = os.path.getsize(archivo_parcial) / (1024 * 1024)
            print(f"   ✅ Guardado: {archivo_parcial} ({tamaño_mb:.2f} MB)")
            
            archivos_parciales.append(archivo_parcial)
            
            # LIBERAR MEMORIA
            del df_lote, dfs_procesados
            gc.collect()
            print(f"   🧹 Memoria liberada")
    
    # ========================================================================
    # FASE 3: COMBINACIÓN FINAL DE LOTES
    # ========================================================================
    print(f"\n{'='*80}")
    print(f"🔗 FASE 3: COMBINACIÓN FINAL")
    print(f"{'='*80}")
    
    print(f"\n📦 Archivos parciales generados: {len(archivos_parciales)}")
    
    if len(archivos_parciales) == 0:
        print("❌ No hay archivos parciales para combinar")
        return None
    
    if len(archivos_parciales) == 1:
        # Si solo hay un lote, renombrar directamente
        print(f"✅ Solo un lote generado, renombrando...")
        os.rename(archivos_parciales[0], ruta_salida)
        df_final = pd.read_parquet(ruta_salida)
    else:
        # Combinar archivos parciales
        print(f"🔄 Combinando {len(archivos_parciales)} archivos parciales...")
        
        # Leer y combinar de forma eficiente
        dfs_parciales = []
        for i, archivo in enumerate(archivos_parciales, 1):
            print(f"   📂 Leyendo lote {i}/{len(archivos_parciales)}...")
            df_temp = pd.read_parquet(archivo)
            dfs_parciales.append(df_temp)
        
        print(f"   🔗 Concatenando todos los lotes...")
        df_final = pd.concat(dfs_parciales, ignore_index=True, sort=False)
        
        # Limpiar
        del dfs_parciales
        gc.collect()
        
        print(f"   💾 Guardando archivo final...")
        df_final.to_parquet(ruta_salida, index=False, engine='pyarrow', compression='snappy')
    
    # Limpiar archivos temporales
    print(f"\n🧹 Limpiando archivos temporales...")
    for archivo in archivos_parciales:
        try:
            if os.path.exists(archivo):
                os.remove(archivo)
                print(f"   ✅ Eliminado: {archivo}")
        except Exception as e:
            print(f"   ⚠️  No se pudo eliminar {archivo}: {e}")
    
    # ========================================================================
    # ESTADÍSTICAS FINALES
    # ========================================================================
    tamaño_final_mb = os.path.getsize(ruta_salida) / (1024 * 1024)
    
    print(f"\n{'='*80}")
    print(f"✅ ¡PROCESO COMPLETADO EXITOSAMENTE!")
    print(f"{'='*80}")
    print(f"📊 DATAFRAME FINAL:")
    print(f"   📏 Dimensiones: {df_final.shape}")
    print(f"   📋 Columnas: {len(df_final.columns)}")
    print(f"   📄 Registros: {len(df_final):,}")
    print(f"   💾 Tamaño archivo: {tamaño_final_mb:.2f} MB")
    print(f"   📁 Ubicación: {ruta_salida}")
    
    # Estadísticas de completitud
    print(f"\n📈 COMPLETITUD:")
    total_celdas = df_final.shape[0] * df_final.shape[1]
    total_nulos = df_final.isnull().sum().sum()
    porcentaje_datos = ((total_celdas - total_nulos) / total_celdas) * 100
    
    print(f"   • Celdas totales: {total_celdas:,}")
    print(f"   • Con datos: {porcentaje_datos:.2f}%")
    print(f"   • NULL: {100-porcentaje_datos:.2f}%")
    
    print(f"\n{'='*80}\n")
    
    return df_final

### 2.5. Función para guardar dataframe en formato parquet

In [7]:
# ============================================================================
# FUNCIÓN 3: GUARDAR DATAFRAME EN FORMATO PARQUET
# ============================================================================

def guardar_dataframe_parquet(df, nombre_archivo, ruta_carpeta="data/processed"):
    """
    Guarda un DataFrame en formato Parquet en la carpeta especificada.
    Incluye validación y limpieza de tipos de datos.
    
    Parámetros:
    -----------
    df : DataFrame
        DataFrame a guardar
    nombre_archivo : str
        Nombre del archivo (sin extensión o con extensión .parquet)
    ruta_carpeta : str
        Ruta de la carpeta donde se guardará (por defecto "data/processed")
    
    Retorna:
    --------
    bool : True si se guardó exitosamente, False en caso contrario
    """
    try:
        print(f"\n{'='*80}")
        print(f"💾 PREPARANDO GUARDADO DE ARCHIVO")
        print(f"{'='*80}")
        
        # Crear copia para no modificar el original
        df_guardar = df.copy()
        
        # ====================================================================
        # PASO 1: DIAGNÓSTICO DE TIPOS DE DATOS
        # ====================================================================
        print(f"\n🔍 PASO 1: Diagnosticando tipos de datos...")
        
        tipos_columnas = df_guardar.dtypes.value_counts()
        print(f"\n   Distribución de tipos:")
        for tipo, cantidad in tipos_columnas.items():
            print(f"   • {tipo}: {cantidad} columnas")
        
        # Identificar columnas problemáticas (tipo object)
        columnas_object = df_guardar.select_dtypes(include=['object']).columns.tolist()
        if columnas_object:
            print(f"\n   ⚠️  {len(columnas_object)} columnas tipo 'object' detectadas")
            print(f"      (pueden causar problemas al guardar)")
        
        # ====================================================================
        # PASO 2: LIMPIEZA Y CONVERSIÓN DE TIPOS
        # ====================================================================
        print(f"\n🔧 PASO 2: Limpiando y convirtiendo tipos de datos...")
        
        # Lista de columnas que DEBEN ser numéricas
        columnas_numericas = [
            'ANO', 'MES', 'HORA', 'MINUTOS', 'COD_DPTO', 'COD_MUNIC',
            'CODPTORE', 'CODMUNRE', 'COD_INST', 'EDAD_MADRE',
            'N_HIJOSV', 'N_HIJOSM', 'PESO_NAC', 'T_GES',
            'CODPRES', 'CODPAISNACFAL', 'CODPAISNACMAD', 'CODOCUR', 'CODMUNOC'
        ]
        
        conversiones_exitosas = 0
        conversiones_fallidas = []
        
        for col in columnas_numericas:
            if col in df_guardar.columns:
                try:
                    tipo_actual = df_guardar[col].dtype
                    
                    # Si ya es numérico, continuar
                    if pd.api.types.is_numeric_dtype(df_guardar[col]):
                        conversiones_exitosas += 1
                        continue
                    
                    # Intentar conversión a numérico
                    df_guardar[col] = pd.to_numeric(df_guardar[col], errors='coerce')
                    conversiones_exitosas += 1
                    
                except Exception as e:
                    conversiones_fallidas.append((col, str(e)))
        
        if conversiones_exitosas > 0:
            print(f"   ✅ {conversiones_exitosas} columnas numéricas procesadas")
        
        if conversiones_fallidas:
            print(f"   ⚠️  {len(conversiones_fallidas)} conversiones fallidas:")
            for col, error in conversiones_fallidas[:5]:
                print(f"      • {col}: {error}")
        
        # Convertir columnas object a string
        columnas_convertidas = 0
        for col in columnas_object:
            try:
                if col in df_guardar.columns:
                    # Convertir a string
                    df_guardar[col] = df_guardar[col].astype(str)
                    # Limpiar valores 'nan' string
                    df_guardar[col] = df_guardar[col].replace(['nan', 'None', '<NA>'], None)
                    columnas_convertidas += 1
            except Exception as e:
                print(f"   ⚠️  Error convirtiendo '{col}': {e}")
        
        if columnas_convertidas > 0:
            print(f"   ✅ {columnas_convertidas} columnas text/string procesadas")
        
        # ====================================================================
        # PASO 3: VALIDACIÓN FINAL
        # ====================================================================
        print(f"\n✓ PASO 3: Validación final...")
        
        # Verificar columnas con tipos mixtos
        columnas_problematicas = []
        for col in df_guardar.columns:
            if df_guardar[col].dtype == 'object':
                # Verificar si hay tipos mixtos en la columna
                tipos_unicos = df_guardar[col].dropna().apply(type).unique()
                if len(tipos_unicos) > 1:
                    columnas_problematicas.append((col, tipos_unicos))
        
        if columnas_problematicas:
            print(f"   ⚠️  {len(columnas_problematicas)} columnas con tipos mixtos detectadas")
            print(f"      Convirtiendo todas a string...")
            
            for col, tipos in columnas_problematicas[:10]:
                try:
                    df_guardar[col] = df_guardar[col].astype(str).replace('nan', None)
                except:
                    pass
        
        print(f"   ✅ Validación completada")
        
        # ====================================================================
        # PASO 4: GUARDAR ARCHIVO
        # ====================================================================
        print(f"\n💾 PASO 4: Guardando archivo...")
        
        # Crear la carpeta si no existe
        os.makedirs(ruta_carpeta, exist_ok=True)
        
        # Asegurar extensión .parquet
        if not nombre_archivo.endswith('.parquet'):
            nombre_archivo += '.parquet'
        
        ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)
        
        # Intentar guardar
        df_guardar.to_parquet(
            ruta_completa,
            index=False,
            engine='pyarrow',
            compression='snappy'
        )
        
        # Obtener tamaño del archivo
        tamaño_mb = os.path.getsize(ruta_completa) / (1024 * 1024)
        
        print(f"\n{'='*80}")
        print(f"✅ ARCHIVO GUARDADO EXITOSAMENTE")
        print(f"{'='*80}")
        print(f"📁 Ubicación: {ruta_completa}")
        print(f"💾 Tamaño: {tamaño_mb:.2f} MB")
        print(f"📊 Registros: {len(df_guardar):,}")
        print(f"📋 Columnas: {len(df_guardar.columns)}")
        
        # Resumen de tipos finales
        print(f"\n📊 Tipos de datos finales:")
        tipos_finales = df_guardar.dtypes.value_counts()
        for tipo, cantidad in tipos_finales.items():
            print(f"   • {tipo}: {cantidad} columnas")
        
        print(f"{'='*80}\n")
        
        return True
    
    except Exception as e:
        print(f"\n{'='*80}")
        print(f"❌ ERROR AL GUARDAR EL ARCHIVO")
        print(f"{'='*80}")
        print(f"Error: {e}")
        
        # Diagnóstico adicional
        print(f"\n🔍 Diagnóstico del error:")
        
        # Identificar la columna problemática si es posible
        if "Conversion failed for column" in str(e):
            columna_error = str(e).split("column ")[1].split(" ")[0]
            print(f"   • Columna problemática: {columna_error}")
            
            if columna_error in df_guardar.columns:
                print(f"   • Tipo actual: {df_guardar[columna_error].dtype}")
                print(f"   • Valores únicos (primeros 10):")
                valores_unicos = df_guardar[columna_error].dropna().unique()[:10]
                for val in valores_unicos:
                    print(f"      - {val} (tipo: {type(val).__name__})")
                
                # Intentar solución automática
                print(f"\n   🔧 Intentando conversión forzada a string...")
                try:
                    df_temp = df_guardar.copy()
                    df_temp[columna_error] = df_temp[columna_error].astype(str).replace('nan', None)
                    
                    ruta_completa = os.path.join(ruta_carpeta, nombre_archivo)
                    df_temp.to_parquet(ruta_completa, index=False, engine='pyarrow', compression='snappy')
                    
                    print(f"   ✅ Guardado exitoso después de conversión")
                    return True
                except Exception as e2:
                    print(f"   ❌ Conversión fallida: {e2}")
        
        print(f"\n💡 Sugerencias:")
        print(f"   1. Verificar tipos de datos en columnas object")
        print(f"   2. Usar df.info() para inspeccionar el DataFrame")
        print(f"   3. Convertir manualmente columnas problemáticas a string")
        print(f"{'='*80}\n")
        
        return False

### 2.6. Función para uso completo

In [8]:
# ============================================================================
# EJEMPLO DE USO COMPLETO
# ============================================================================

def ejemplo_uso_completo():
    """
    Ejemplo completo del flujo de trabajo para homologar y combinar defunciones.
    """
    
    print("="*80)
    print("PROCESO DE HOMOLOGACIÓN Y COMBINACIÓN DE DEFUNCIONES")
    print("="*80)
    
    # PASO 1: Cargar diccionario de homologación
    print("\n📖 PASO 1: Cargando diccionario de homologación...")
    ruta_excel = "ruta/al/archivo/homologacion.xlsx"  # AJUSTAR RUTA
    diccionario, id_estandar = cargar_diccionario_homologacion(ruta_excel, "Campos Defunciones")
    
    if diccionario is None:
        print("❌ No se pudo continuar sin el diccionario")
        return
    
    # PASO 2: Definir lista de DataFrames a procesar
    print("\n🗂️ PASO 2: Definiendo DataFrames a procesar...")
    
    # Lista de tuplas: (nombre_variable, id_correspondiente)
    lista_dataframes = [
        ('defunciones_1979_1991_procesado', 1),
        ('defunciones_1992_1996_procesado', 2),
        ('defunciones_1997_1997_procesado', 3),
        ('defunciones_1998_2007_procesado', 4),
        ('defunciones_2008_2011_procesado', 5),
        ('defunciones_2012_2013_procesado', 6),
        ('defunciones_2014_procesado', 7),
        ('defunciones_2015_procesado', 8),
        ('defunciones_2016_procesado', 9),
        ('defunciones_2017_procesado', 10),
        ('defunciones_2018_procesado', 10),  # Mismo ID que 2017
        ('defunciones_2019_procesado', 12),
        ('defunciones_2020_procesado', 13),
        ('defunciones_2021_procesado', 14),
        ('defunciones_2022_procesado', 15),
        ('defunciones_2023_procesado', 16),
        ('defunciones_2024_procesado', 17)
    ]
    
    # PASO 3: Homologar y combinar
    print("\n🔄 PASO 3: Homologando y combinando DataFrames...")
    defunciones = homologar_y_combinar_defunciones(lista_dataframes, diccionario, id_estandar)
    
    if defunciones is None:
        print("❌ No se pudo crear el DataFrame combinado")
        return
    
    # PASO 4: Guardar resultado
    print("\n💾 PASO 4: Guardando DataFrame combinado...")
    exito = guardar_dataframe_parquet(defunciones, "defunciones_completo", "data/processed")
    
    if exito:
        print("\n" + "="*80)
        print("✅ PROCESO COMPLETADO EXITOSAMENTE")
        print("="*80)
        
        # Mostrar muestra del DataFrame
        print("\n📋 Muestra del DataFrame combinado (primeras 5 filas):")
        print(defunciones.head())
    
    return defunciones


### 2.6. Función convertir lista de archivos a tuplas

In [9]:
# ============================================================================
# FUNCIÓN AUXILIAR: CONVERTIR LISTA DE ARCHIVOS A FORMATO REQUERIDO
# ============================================================================

def convertir_lista_archivos_a_tuplas(lista_archivos):
    """
    Convierte una lista de nombres de archivos .parquet a una lista de tuplas
    con formato (nombre_variable, id_incremental) para usar en homologación.
    
    CONSIDERACIÓN ESPECIAL: Los archivos de 2017 y 2018 comparten el ID 10
    
    Parámetros:
    -----------
    lista_archivos : list
        Lista de nombres de archivos .parquet
        Ejemplo: ["defunciones_2017_procesado.parquet", "defunciones_2018_procesado.parquet"]
    
    Retorna:
    --------
    list of tuples : Lista de tuplas (nombre_variable, id)
    
    Ejemplo de uso:
    ---------------
    archivos_a_leer = [
        "defunciones_2017_procesado.parquet",
        "defunciones_2018_procesado.parquet"
    ]
    lista_tuplas = convertir_lista_archivos_a_tuplas(archivos_a_leer)
    """
    lista_tuplas = []
    
    for idx, archivo in enumerate(lista_archivos, start=1):
        # Remover la extensión .parquet
        nombre_variable = os.path.splitext(archivo)[0]
        
        # ID especial para 2017 y 2018: ambos usan ID 10
        if '2017' in nombre_variable or '2018' in nombre_variable:
            id_asignado = 10
        else:
            id_asignado = idx
        
        # Crear tupla (nombre_variable, id)
        lista_tuplas.append((nombre_variable, id_asignado))
    
    print(f"✅ Conversión completada:")
    print(f"   📁 Archivos procesados: {len(lista_tuplas)}")
    print(f"   🆔 IDs asignados: 1 a {len(lista_tuplas)}")
    print(f"   ⚠️  NOTA: 2017 y 2018 comparten ID 10")
    
    # Mostrar los primeros 3 y últimos 3 para verificación
    print(f"\n📋 Primeros 3 elementos:")
    for i in range(min(3, len(lista_tuplas))):
        print(f"   {lista_tuplas[i]}")
    
    if len(lista_tuplas) > 6:
        print(f"   ...")
        print(f"📋 Últimos 3 elementos:")
        for i in range(max(0, len(lista_tuplas) - 3), len(lista_tuplas)):
            print(f"   {lista_tuplas[i]}")
    
    return lista_tuplas

### 2.7. Función para optimizar tipos de dataframe

In [10]:
def optimizar_tipos_dataframe(df):
    """
    Optimiza tipos de datos para reducir uso de memoria.
    CORREGIDO: Maneja correctamente valores None/NaN para evitar errores de PyArrow.
    """
    # Columnas numéricas que deberían ser enteros
    columnas_int = [
        'ANO', 'MES', 'HORA', 'MINUTOS', 'COD_DPTO', 'COD_MUNIC',
        'CODPTORE', 'CODMUNRE', 'EDAD_MADRE', 'N_HIJOSV', 'N_HIJOSM'
    ]
    
    # Columnas numéricas que pueden ser float
    columnas_float = ['PESO_NAC', 'T_GES']
    
    for col in columnas_int:
        if col in df.columns:
            try:
                # Convertir a numérico
                df[col] = pd.to_numeric(df[col], errors='coerce')
                # Usar Int32 (permite NaN y ahorra memoria)
                if df[col].notna().any():
                    df[col] = df[col].astype('Int32')
            except:
                pass
    
    for col in columnas_float:
        if col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                df[col] = df[col].astype('float32')
            except:
                pass
    
    # =========================================================================
    # CRÍTICO: Optimizar columnas de texto SIN crear strings 'None'
    # =========================================================================
    columnas_object = df.select_dtypes(include=['object']).columns
    for col in columnas_object:
        try:
            # PASO 1: Reemplazar valores None/NaN con pd.NA ANTES de convertir a string
            df[col] = df[col].fillna(pd.NA)
            
            # PASO 2: Convertir solo valores no-NA a string
            mask_not_na = df[col].notna()
            if mask_not_na.any():
                df.loc[mask_not_na, col] = df.loc[mask_not_na, col].astype(str)
            
            # PASO 3: Limpiar strings problemáticos y volver a None
            df[col] = df[col].replace(['nan', 'None', '<NA>', 'NaN', 'null'], None)
            
            # PASO 4: Si tiene pocas categorías únicas, usar category
            n_unique = df[col].nunique()
            if n_unique < len(df) * 0.5 and n_unique > 0:
                df[col] = df[col].astype('category')
        except Exception as e:
            # Si falla, intentar conversión simple
            try:
                df[col] = df[col].astype(str)
                df[col] = df[col].replace(['nan', 'None', '<NA>'], None)
            except:
                pass
    
    return df

### 2.8. Función para limpiar memoria ram

In [11]:
def limpiar_memoria():
    """Libera memoria RAM y limpia objetos no usados."""
    gc.collect()
    process = psutil.Process(os.getpid())
    memoria_libre = process.memory_info().rss / (1024 ** 3)
    print(f"🔄 Memoria usada antes de limpieza: {memoria_libre:.2f} GB")

    for var in list(globals().keys()):
        if not var.startswith("_") and var not in ["os", "gc", "psutil", "limpiar_memoria"]:
            del globals()[var]

    gc.collect()
    memoria_final = process.memory_info().rss / (1024 ** 3)
    print(f"✅ Memoria usada después de limpieza: {memoria_final:.2f} GB")

### 2.9. Función para diagnosticar duplicados de columnas

In [12]:
def diagnosticar_duplicados_columnas(lista_dataframes, diccionario_homolog):
    """
    Función de diagnóstico: identifica qué DataFrames tienen columnas duplicadas
    después de aplicar el diccionario de homologación.
    """
    print(f"\n{'='*80}")
    print(f"🔍 DIAGNÓSTICO DE COLUMNAS DUPLICADAS")
    print(f"{'='*80}\n")
    
    problemas_encontrados = []
    
    for idx, (nombre_var, id_archivo) in enumerate(lista_dataframes, 1):
        if nombre_var not in globals():
            continue
        
        df = globals()[nombre_var].copy()
        
        # Aplicar renombramiento
        if id_archivo in diccionario_homolog:
            mapeo = diccionario_homolog[id_archivo]
            for col_antigua, col_nueva in mapeo.items():
                if col_antigua in df.columns:
                    df.rename(columns={col_antigua: col_nueva}, inplace=True)
        
        # Verificar duplicados
        duplicados = df.columns[df.columns.duplicated()].unique()
        
        if len(duplicados) > 0:
            print(f"⚠️  [{idx}] {nombre_var} (ID: {id_archivo})")
            print(f"   Columnas duplicadas: {list(duplicados)}")
            
            # Mostrar todas las apariciones
            for col_dup in duplicados:
                indices = [i for i, col in enumerate(df.columns) if col == col_dup]
                print(f"   '{col_dup}' aparece en posiciones: {indices}")
            
            problemas_encontrados.append((nombre_var, id_archivo, list(duplicados)))
            print()
    
    if not problemas_encontrados:
        print("✅ No se encontraron columnas duplicadas en ningún DataFrame")
    else:
        print(f"\n{'='*80}")
        print(f"📊 RESUMEN: {len(problemas_encontrados)} DataFrames con duplicados")
        print(f"{'='*80}")
        
        for nombre, id_df, dups in problemas_encontrados:
            print(f"• {nombre} (ID {id_df}): {len(dups)} duplicados")
    
    return problemas_encontrados

### 2.10. Función para limpiar dataframe para parquet

In [13]:
def limpiar_dataframe_para_parquet(df):
    """
    Limpia un DataFrame para asegurar compatibilidad con PyArrow/Parquet.
    """
    df_limpio = df.copy()
    
    for col in df_limpio.columns:
        dtype = df_limpio[col].dtype
        
        # Para columnas object, asegurar que no hay strings 'None'
        if dtype == 'object' or str(dtype) == 'string':
            # Reemplazar strings problemáticos con None real
            df_limpio[col] = df_limpio[col].replace(
                ['None', 'nan', 'NaN', '<NA>', 'null', 'NULL'], 
                None
            )
            
            # Si todos son None o vacíos, convertir a string explícitamente
            if df_limpio[col].notna().sum() == 0:
                df_limpio[col] = df_limpio[col].astype(str)
            else:
                # Intentar convertir a string solo valores no-None
                mask = df_limpio[col].notna()
                if mask.any():
                    try:
                        df_limpio.loc[mask, col] = df_limpio.loc[mask, col].astype(str)
                    except:
                        df_limpio[col] = df_limpio[col].astype(str)
                        df_limpio[col] = df_limpio[col].replace('None', None)
    
    return df_limpio

## 3. PROCESAMIENTO FUNCIONES

### 3.1. Ejecutar Combinación y almacenamiento

In [ ]:
# 1. Limpiar memoria
gc.collect()

# 2. CARGAR ARCHIVOS
archivos_a_leer = [
    "defunciones_1979_1991_procesado.parquet",
    "defunciones_1992_1996_procesado.parquet",
    "defunciones_1997_1997_procesado.parquet",
    "defunciones_1998_2007_procesado.parquet",
    "defunciones_2008_2011_procesado.parquet",
    "defunciones_2012_2013_procesado.parquet",
    "defunciones_2014_procesado.parquet",
    "defunciones_2015_procesado.parquet",
    "defunciones_2016_procesado.parquet",
    "defunciones_2017_procesado.parquet",
    "defunciones_2018_procesado.parquet",
    "defunciones_2019_procesado.parquet",
    "defunciones_2020_procesado.parquet",
    "defunciones_2021_procesado.parquet",
    "defunciones_2022_procesado.parquet",
    "defunciones_2023_procesado.parquet",
    "defunciones_2024_procesado.parquet"
]
cargar_archivos_seleccionados_con_id("data/processed", archivos_a_leer)

# 3. Convertir lista
lista_dataframes = convertir_lista_archivos_a_tuplas(archivos_a_leer)

# 4. Cargar diccionario
diccionario, id_estandar = cargar_diccionario_homologacion(
    "data/raw/Referenciales/Fuentes de Información Recolección Inicial.xlsx", 
    "Campos Defunciones"
)

# 5. DIAGNÓSTICO - CORREGIDO: solo 2 argumentos
print("\n🔍 EJECUTANDO DIAGNÓSTICO...")
problemas = diagnosticar_duplicados_columnas(lista_dataframes, diccionario)

# Si no hay problemas, proceder
if not problemas:
    print("\n✅ No se detectaron problemas, procediendo con el procesamiento...")
    
    defunciones = homologar_y_combinar_defunciones_optimizado(
        lista_dataframes, 
        diccionario, 
        id_estandar,
        ruta_salida="data/processed/defunciones_completo.parquet",
        batch_size=1
    )
    
    print("✅ Proceso completado!")
else:
    print("\n⚠️ Se detectaron problemas. Revisa el diagnóstico antes de continuar.")
    print("💡 Si los duplicados son esperados (misma columna con diferentes nombres),")
    print("   el proceso los consolidará automáticamente.")
    
    respuesta = input("\n¿Deseas continuar de todos modos? (s/n): ")
    if respuesta.lower() == 's':
        defunciones = homologar_y_combinar_defunciones_optimizado(
            lista_dataframes, 
            diccionario, 
            id_estandar,
            ruta_salida="data/processed/defunciones_completo.parquet",
            batch_size=1
        )
        print("✅ Proceso completado!")


✅ defunciones_1979_1991_procesado cargado con ID = 1 (1869025 filas)
✅ defunciones_1992_1996_procesado cargado con ID = 2 (848360 filas)
✅ defunciones_1997_1997_procesado cargado con ID = 3 (170753 filas)
✅ defunciones_1998_2007_procesado cargado con ID = 4 (1886949 filas)
✅ defunciones_2008_2011_procesado cargado con ID = 5 (790223 filas)
✅ defunciones_2012_2013_procesado cargado con ID = 6 (402827 filas)
✅ defunciones_2014_procesado cargado con ID = 7 (210051 filas)
✅ defunciones_2015_procesado cargado con ID = 8 (219472 filas)
✅ defunciones_2016_procesado cargado con ID = 9 (223078 filas)
✅ defunciones_2017_procesado cargado con ID = 10 (227624 filas)
✅ defunciones_2018_procesado cargado con ID = 11 (236932 filas)
✅ defunciones_2019_procesado cargado con ID = 12 (244355 filas)
✅ defunciones_2020_procesado cargado con ID = 13 (300853 filas)
✅ defunciones_2021_procesado cargado con ID = 14 (363089 filas)
✅ defunciones_2022_procesado cargado con ID = 15 (287251 filas)
✅ defunciones_202

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = None
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = None
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(ax

   🔧 Optimizando tipos de datos...
   ✅ Procesado completo: (1869025, 205)

🔄 [2/17] defunciones_1992_1996_procesado (ID: 2)
   📏 Dimensiones originales: (848360, 35)
   📋 Columnas originales: 35
   🔄 Aplicando mapeo universal de nombres...
   ✅ Renombrando 1 columnas:
      • 'PERMAN_MUN' → 'TIEM_PER'
   📋 Columnas después de renombrar: 34
   ➕ Agregando 171 columnas faltantes


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = None
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = None
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(ax

   🔧 Optimizando tipos de datos...
   ✅ Procesado completo: (848360, 205)

🔗 Combinando 2 DataFrames del lote...
   🔍 Verificando consistencia de columnas...
   ✅ Columnas comunes: 205


...

   🔧 Optimizando tipos de datos...
   ✅ Procesado completo: (170753, 205)

🔄 [4/17] defunciones_1998_2007_procesado (ID: 4)
   📏 Dimensiones originales: (1886949, 97)
   📋 Columnas originales: 97
   🔄 Aplicando mapeo universal de nombres...
   ✅ Renombrando 2 columnas:
      • 'PMAN_MUER' → 'P_PMAN_IRIS'
      • 'C_ANT1' → 'CAUSA_MULT'
   📋 Columnas después de renombrar: 96
   ➕ Agregando 109 columnas faltantes


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = None
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[col] = None
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:229: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(ax

   🔧 Optimizando tipos de datos...
   ✅ Procesado completo: (1886949, 205)

🔗 Combinando 2 DataFrames del lote...
   🔍 Verificando consistencia de columnas...
   ✅ Columnas comunes: 205


C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:267: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_lote = pd.concat(dfs_procesados, ignore_index=True, sort=False)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:267: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_lote = pd.concat(dfs_procesados, ignore_index=True, sort=False)
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_4284\1181406155.py:267: FutureWarning: The behavior of DataFrame concatenation with empt

   ✅ Lote combinado: (2057702, 205)
   💾 Guardando lote parcial...
